# ECG Baseline Model
An ECG-only 1D CNN for HFrEF prediction. The processed data is split at the subject level to prevent data leakage and handle class imbalance using a weighted loss function.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_curve, roc_auc_score, precision_recall_curve, average_precision_score
import matplotlib.pyplot as plt
from tqdm import tqdm

sys.path.append(os.path.abspath('..'))
from src.data_loader import get_dataloaders
from src.ecg_model import ECG_Encoder

metadata = pd.read_csv('../data/processed/processed_metadata.csv')

# split data ensuring segments from the same Record_ID stay together to prevent leakage
gss = GroupShuffleSplit(n_splits=1, train_size=0.8, random_state=42)
train_idx, val_idx = next(gss.split(metadata, groups=metadata['Record_ID']))

train_df = metadata.iloc[train_idx]
val_df = metadata.iloc[val_idx]

print(f"Training segments: {len(train_df)}")
print(f"Validation segments: {len(val_df)}")

## Initialize Dataloaders and Model
PyTorch `.pt` tensors are directly loaded from the disk.

In [ ]:
PROCESSED_DIR = '../data/processed/'

# initialize custom dataloaders
train_loader, val_loader = get_dataloaders(train_df, val_df, data_dir=PROCESSED_DIR, batch_size=16)

# setup device for GPU acceleration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# initialize model
ecg_model = ECG_Encoder().to(device)

print(f"DataLoaders ready. Training batches: {len(train_loader)}")
print(f"Model initialized on device: {device}")

## Training Loop
Binary Cross Entropy with Logits Loss (`BCEWithLogitsLoss`) is used. Since we have few HFrEF subjects (1760 Normal to 240 HFrEF), a positive weight of ~7.3 is applied to the loss function to penalize the model heavily if it misses an HFrEF case.

In [ ]:
os.makedirs('../models', exist_ok=True) # create models directory if it doesn't exist

# Manually set hyperparameters
num_epochs = 10
learning_rate = 1e-3

pos_weight = torch.tensor([7.3]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(ecg_model.parameters(), lr=learning_rate)

best_auroc = 0.0
MODEL_SAVE_PATH = '../models/best_ecg_model.pt'

for epoch in range(num_epochs):
    # training
    ecg_model.train()
    train_loss = 0.0
    
    for ecg, pcg, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]"):
        ecg = ecg.to(device)
        labels = labels.to(device).unsqueeze(1)
        
        optimizer.zero_grad()
        outputs = ecg_model(ecg)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * ecg.size(0)
        
    epoch_train_loss = train_loss / len(train_loader.dataset)
    
    # validation
    ecg_model.eval()
    val_loss = 0.0
    all_labels = []
    all_preds = []
    
    with torch.no_grad():
        for ecg, pcg, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]"):
            ecg = ecg.to(device)
            labels = labels.to(device).unsqueeze(1)
            
            outputs = ecg_model(ecg)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * ecg.size(0)
            
            # sigmoid to convert raw logits to probabilities
            preds = torch.sigmoid(outputs)
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            
    epoch_val_loss = val_loss / len(val_loader.dataset)
    
    # calculate metrics
    auroc = roc_auc_score(all_labels, all_preds)
    auprc = average_precision_score(all_labels, all_preds)
    
    print(f"Epoch {epoch+1} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | AUROC: {auroc:.4f} | AUPRC: {auprc:.4f}")
    
    # save model if AUROC improves
    if auroc > best_auroc:
        print(f"*** Validation AUROC improved from {best_auroc:.4f} to {auroc:.4f}. Saving model! ***")
        best_auroc = auroc
        # We save state_dict() instead of the whole model to ensure flexibility when loading
        torch.save(ecg_model.state_dict(), MODEL_SAVE_PATH)

## Model Evaluation

In [ ]:
# load the best saved weights
ecg_model.load_state_dict(torch.load(MODEL_SAVE_PATH))
ecg_model.eval()

# gather predictions on validation set
all_labels = []
all_preds = []
with torch.no_grad():
    for ecg, _, labels in val_loader:
        ecg = ecg.to(device)
        preds = torch.sigmoid(ecg_model(ecg))
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

# calculate optimal threshold (F1-Score)
precisions, recalls, thresholds_pr = precision_recall_curve(all_labels, all_preds)

# calculate F1 for all thresholds (adding 1e-8 to avoid division by zero)
f1_scores = (2 * precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-8)
optimal_idx = np.argmax(f1_scores)
optimal_threshold = thresholds_pr[optimal_idx]

print(f"Optimal Probability Cutoff: {optimal_threshold:.4f}")
print(f"Max F1-Score: {f1_scores[optimal_idx]:.4f}")

# calculate ROC Curve data
fpr, tpr, thresholds_roc = roc_curve(all_labels, all_preds)

# plotting
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ROC Curve (AUROC)
axes[0].plot(fpr, tpr, color='blue', lw=2)
axes[0].plot([0, 1], [0, 1], color='black', linestyle='--')
axes[0].set_xlabel('False Positive Rate (False Alarms)')
axes[0].set_ylabel('True Positive Rate (Caught Cases)')
axes[0].set_title('ROC Curve')

# Precision-Recall Curve (AUPRC)
axes[1].plot(recalls[:-1], precisions[:-1], color='green', lw=2)
# Mark the optimal threshold on the PR curve
axes[1].plot(recalls[optimal_idx], precisions[optimal_idx], marker='o', markersize=8, color="red", label="Optimal Threshold")
axes[1].set_xlabel('Recall (Caught Cases)')
axes[1].set_ylabel('Precision (Accuracy of Alarms)')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend()

plt.tight_layout()
plt.show()